In [1]:
import time
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from skimage.transform import resize
from skimage import feature
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

In [2]:
annotations_dir = 'dataset/annotations'
images_dir = 'dataset/images'

In [3]:
img_list = []
label_list = []
for xml_file in os.listdir(annotations_dir):
    xml_filepath = os.path.join(annotations_dir, xml_file)
    tree = ET.parse(xml_filepath)
    root = tree.getroot()

    folder = root.find('folder').text
    filename = root.find('filename').text
    path = os.path.join(images_dir, filename)
    img = cv2.imread(path)

    for obj in root.findall('object'):
        class_name = obj.find('name').text
        if class_name == 'trafficlight':
            continue

        xmin = int(obj.find('bndbox/xmin').text)
        ymin = int(obj.find('bndbox/ymin').text)
        xmax = int(obj.find('bndbox/xmax').text)
        ymax = int(obj.find('bndbox/ymax').text)

        object_img = img[ymin:ymax, xmin:xmax]
        img_list.append(object_img)
        label_list.append(class_name)

# Preprocess Image

In [4]:
def preprocess_img(img):
    if len(img.shape) > 2:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img = img.astype(np.float32)

    resized_img = resize(img, (64, 64), anti_aliasing=True)

    hog_features = feature.hog(
        resized_img, 
        orientations=9, 
        pixels_per_cell=(8, 8), 
        cells_per_block=(2, 2), 
        transform_sqrt=True, 
        block_norm='L2-Hys'
    )
    return hog_features

In [5]:
img_features = []
for img in img_list:
    img_features.append(preprocess_img(img))

In [6]:
img_features

[array([0.00141404, 0.00018905, 0.00067764, ..., 0.02802324, 0.06912123,
        0.07639673], shape=(1764,), dtype=float32),
 array([0.01298072, 0.00667271, 0.00194479, ..., 0.03192824, 0.02022759,
        0.02473512], shape=(1764,), dtype=float32),
 array([0.0057954 , 0.00097038, 0.00057083, ..., 0.00140552, 0.        ,
        0.00232025], shape=(1764,), dtype=float32),
 array([0.00208534, 0.01059551, 0.36030295, ..., 0.0029671 , 0.00148393,
        0.00125475], shape=(1764,), dtype=float32),
 array([0.00938648, 0.        , 0.01965713, ..., 0.010738  , 0.        ,
        0.02415599], shape=(1764,), dtype=float32),
 array([0.01368523, 0.        , 0.02140873, ..., 0.        , 0.        ,
        0.        ], shape=(1764,), dtype=float32),
 array([0.01255343, 0.00650927, 0.00575718, ..., 0.04690857, 0.01377166,
        0.03081629], shape=(1764,), dtype=float32),
 array([0.00694832, 0.01661516, 0.37902093, ..., 0.02807329, 0.01804562,
        0.        ], shape=(1764,), dtype=float32),


# Traffic Sign Classification

In [7]:
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(label_list)

## Create Train/Test Dataset

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    list(img_features), 
    encoded_labels, 
    test_size=0.2, 
    random_state=42)

In [9]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Train and Evaluate

In [10]:
clf = SVC(kernel='rbf', C=0.5)
clf.fit(X_train, y_train)

SVC(C=0.5)

In [11]:
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
accuracy

0.986046511627907

# Traffic Localization

In [12]:
def sliding_window(image, step_size, window_size):
    windows = []
    for y in range(0, image.shape[0] - window_size[1] + 1, step_size):
        for x in range(0, image.shape[1] - window_size[0] + 1, step_size):
            windows.append((x, y, x + window_size[0], y + window_size[1]))
    return windows


In [13]:
def pyramid(image, scale=1.5, min_size=(30, 30)):
    yield image
    while True:
        w = int(image.shape[1] / scale)
        h = int(image.shape[0] / scale)
        image = resize(image, (h, w), anti_aliasing=True)
        if image.shape[0] < min_size[1] or image.shape[1] < min_size[0]:
            break
        yield image

In [14]:
def visualize_bbox(image, bbox):
    img = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    for box in bbox:
        cv2.rectangle(img, (box[0], box[1]), (box[2], box[3]), (0, 255, 0), 2)
        class_name = label_encoder.inverse_transform([box[4]])[0]
        label = f'{class_name}: {box[5]:.2f}'
        (w,h),_ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
        cv2.rectangle(img, (box[0], box[1]), (box[0]+w, box[1]-h), (0, 255, 0), -1)
        cv2.putText(img, label, (box[0], box[1]), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
    plt.imshow(img)
    plt.show()

In [15]:
def non_max_suppression(boxes, overlap_thresh=0.5):
    if len(boxes) == 0:
        return []
    
    boxes = np.array(boxes)
    pick = []
    x1, y1, x2, y2, scores = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3], boxes[:, 5]

    idxs = np.argsort(scores)[::-1]
    while len(idxs) > 0:
        i = idxs[0]
        pick.append(i)
        xx1 = np.maximum(x1[i], x1[idxs[1:]])
        yy1 = np.maximum(y1[i], y1[idxs[1:]])
        xx2 = np.minimum(x2[i], x2[idxs[1:]])
        yy2 = np.minimum(y2[i], y2[idxs[1:]])

        w = np.maximum(0, xx2 - xx1 + 1)
        h = np.maximum(0, yy2 - yy1 + 1)
        overlap = (w * h) / ((x2[i] - x1[i] + 1) * (y2[i] - y1[i] + 1) + (x2[idxs[1:]] - x1[idxs[1:]] + 1) * (y2[idxs[1:]] - y1[idxs[1:]] + 1) - w * h)

        idxs = idxs[np.where(overlap <= overlap_thresh)[0] + 1]
    return boxes[pick].tolist()


In [16]:
def create_image_pyramid(image, scales=[1.0, 0.75, 0.5, 0.25]):
    pyramid = []
    for scale in scales:
        resized = cv2.resize(image, None, fx=scale, fy=scale, interpolation=cv2.INTER_LINEAR)
        pyramid.append((resized, scale))
    return pyramid

In [17]:
def detect_objects(image, clf, scaler, step_size=32, window_size=(64, 64), visualize=False):
    bboxes = []
    img_height, img_width = image.shape[:2]

    for (x_min, y_min, x_max, y_max) in sliding_window(image, step_size, window_size):
        window_img = image[y_min:ymax, xmin:xmax]
        if window_img.shape[:2] != window_size:
            continue  # Skip incomplete windows at edges

        # Extract and normalize features
        features = preprocess_img(window_img)
        norm_features = scaler.transform([features]).reshape(1, -1)

        # Get decision score
        scores = clf.decision_function(norm_features)
        score = scores[0] if len(scores.shape) == 1 else scores.max()  # Support multi-class

        if score > 0:  # Positive detection
            bboxes.append((x_min, y_min, x_max, y_max, score))

        # Optional visualization
        if visualize:
            vis_image = image.copy()
            cv2.rectangle(vis_image, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)
            cv2.imshow('Sliding Window', vis_image)
            cv2.waitKey(1)

    return bboxes


: 

In [ ]:
# Path to the image
image_path = "dataset/images/road875.png"
if not os.path.exists(image_path):
    raise FileNotFoundError(f"Image not found at {image_path}")

# Load the image
image = cv2.imread(image_path)

# Detect objects
bboxes = detect_objects(image, clf, scaler, visualize=False)

# Apply Non-Max Suppression
filtered_bboxes = non_max_suppression(bboxes)

# Draw detections on the image
if filtered_bboxes:
    for (x_min, y_min, x_max, y_max, score) in filtered_bboxes:
        cv2.rectangle(image, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)
        cv2.putText(image, f"{score:.2f}", (x_min, y_min - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
else:
    print("No objects detected.")

# Save and display the result
output_path = "output/detections_road875.png"
cv2.imwrite(output_path, image)
print(f"Detection result saved to {output_path}")

cv2.imshow("Detections", image)
cv2.waitKey(0)